# Week 02: Mini Semantic Search

Search frequently asked questions (FAQs) by meaning using text embeddings and ChromaDB.

## Objectives
- Load FAQ data from a CSV file.
- Clean and prepare the data.
- Combine questions and answers into text chunks.
- Generate embeddings using Sentence Transformers.
- Store embeddings in ChromaDB.
- Search for the three most relevant FAQs.
- Calculate basic question-length statistics using NumPy.


## Step 1: Load the data

Load the FAQ dataset using Pandas. The dataset should contain 30 FAQs before cleaning.

In [ ]:
import numpy as np
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

df = pd.read_csv("data/faqs.csv")

print(f"Loaded {len(df)} FAQs")
display(df.head())

## Step 2: Clean the data

Remove incomplete records, empty questions and answers, duplicate questions, and unnecessary spaces.

In [ ]:
before = len(df)

df = df.dropna(subset=["question", "answer"]).copy()
df["question"] = df["question"].str.strip()
df["answer"] = df["answer"].str.strip()

df = df[
    (df["question"] != "") &
    (df["answer"] != "")
]

df = df.drop_duplicates(subset=["question"])
df = df.reset_index(drop=True)

print(f"FAQs before cleaning: {before}")
print(f"FAQs after cleaning: {len(df)}")

## Step 3: Create text chunks

Combine each question and its answer into one text chunk. Each chunk represents one FAQ.

In [ ]:
df["chunk"] = (
    "Question: " + df["question"] +
    " Answer: " + df["answer"]
)

chunks = df["chunk"].tolist()

print(f"Created {len(chunks)} chunks")
print("\nExample chunk:")
print(chunks[0])

## Step 4: Generate text embeddings

Sentence Transformers converts each text chunk into a numerical vector. These vectors help the system compare meaning.

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(chunks)

print(f"Number of embeddings: {len(embeddings)}")
print(f"Embedding shape: {embeddings.shape}")

## Step 5: Store embeddings in ChromaDB

Store the text chunks, their embeddings, and the original questions and answers in a ChromaDB collection.

In [ ]:
client = chromadb.Client()
collection = client.create_collection(name="faqs")

collection.add(
    ids=[str(i) for i in range(len(chunks))],
    embeddings=embeddings.tolist(),
    documents=chunks,
    metadatas=[
        {"question": q, "answer": a}
        for q, a in zip(df["question"], df["answer"])
    ]
)

print(f"Stored documents: {collection.count()}")

## Step 6: Create the search function

Convert the user's query into an embedding and ask ChromaDB for the three most relevant FAQ matches.

In [ ]:
def search_faqs(query, n=3):
    query_embedding = model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=min(n, collection.count())
    )

    print(f"\nQuery: {query}")
    print("-" * 60)

    for i, item in enumerate(results["metadatas"][0], start=1):
        print(f"{i}. {item['question']}")
        print(f"   Answer: {item['answer']}")

    return results

## Step 7: Test semantic search

Test the system using different questions. The system should retrieve relevant FAQs based on meaning, not only exact word matches.

In [ ]:
search_faqs("How do I track changes in my code?")
search_faqs("What tool helps applications run everywhere?")
search_faqs("How can I find patterns in data?")
search_faqs("How can I search for information by meaning?")
search_faqs("Which Python library helps me work with arrays?")

## Step 8: Calculate basic statistics

Use NumPy to calculate the average, median, maximum, and 75th percentile of the question lengths.

In [ ]:
lengths = df["question"].str.len().to_numpy()

print(f"Number of FAQs after cleaning: {len(df)}")
print(f"Average question length: {np.mean(lengths):.1f}")
print(f"Median question length: {np.median(lengths):.1f}")
print(f"Maximum question length: {np.max(lengths)}")
print(f"75th percentile: {np.percentile(lengths, 75):.1f}")

## Summary

This project implements a mini semantic search system for frequently asked questions (FAQs).

The system loads FAQ data from a CSV file, cleans the data, combines each question and answer into a text chunk, and generates text embeddings using the `all-MiniLM-L6-v2` model.

The embeddings are stored in ChromaDB, allowing users to search for relevant FAQs based on meaning rather than exact keyword matches.

The project also tests several search queries and uses NumPy to calculate basic statistics about question lengths.

The dataset should contain 30 FAQs before cleaning. The final number may be lower if duplicate or incomplete records are removed.
